# 🧭 Pick your tier — VRAM calculator + model recommender

Not sure where to start? This notebook asks 3 questions, looks at your GPU, and points you to the right tier.

**No GPU needed to run this** — in fact, running it on CPU (or Colab Free with no accelerator) is fine. We're just matching you to a notebook before you spin up the real runtime.

Takes ~60 seconds.

In [ ]:
# Minimal install — only torch (for GPU detection). ipywidgets is optional.
try:
    import torch
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch'], check=True)
    import torch

print(f'torch {torch.__version__} ready')

## Step 1 — detect your GPU

We check `torch.cuda` for a device. If you're on CPU, that's fine — we'll still recommend a tier, you just won't be able to *train* here.

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU detected: {name}')
    print(f'VRAM:         {vram_gb:.1f} GB')
else:
    name = 'CPU'
    vram_gb = 0.0
    print("You're on CPU — use Colab Free T4 runtime (Runtime -> Change runtime type -> T4 GPU)")
    print('We can still recommend a tier below; just run the actual training notebook on a GPU runtime.')
    # Assume T4 (16 GB) for recommendation purposes so the advisor is useful.
    vram_gb = 16.0
    print(f'(Assuming {vram_gb:.0f} GB T4 for the recommendation.)')

## Step 2 — tell us your target

Edit the 3 variables below. Defaults are safe for a first-time user.

In [ ]:
TRAINING_TIME_BUDGET_HOURS = 1.0   # how long can you wait?
MODEL_FAMILY_PREF         = 'any'  # 'gemma' | 'qwen' | 'llama' | 'any'
DIFFICULTY                = 'beginner'  # 'beginner' | 'intermediate' | 'advanced'

print(f'Time budget: {TRAINING_TIME_BUDGET_HOURS} h')
print(f'Family:      {MODEL_FAMILY_PREF}')
print(f'Difficulty:  {DIFFICULTY}')

## Step 3 — the recommendation

We match your VRAM + time budget to a tier and hand you the notebook link. We also compute the mid-layer heuristic and the SAE param budget for you.

In [ ]:
def recommend(vram_gb, time_budget, preference, difficulty):
    """Pick the richest tier that fits both VRAM and time budget."""
    TIERS = [
        dict(name='Tier 1 Hobbyist', vram_min=10, time_hr=0.5, model='Gemma-2-2B',
             notebook='01_hobbyist_gemma2_2b_colab.ipynb', d_model=2304, layer=15,
             cost_usd=0.0, emoji='🌱',
             url='https://colab.research.google.com/github/OpenInterpretability/notebooks/blob/main/notebooks/01_hobbyist_gemma2_2b_colab.ipynb'),
        dict(name='Tier 2 Explorer', vram_min=24, time_hr=4.5, model='Qwen3.5-4B (hybrid GDN)',
             notebook='02_explorer_qwen35_4b_kaggle.ipynb', d_model=2560, layer=18,
             cost_usd=0.0, emoji='🚀',
             url='https://github.com/OpenInterpretability/notebooks/blob/main/notebooks/02_explorer_qwen35_4b_kaggle.ipynb'),
        dict(name='Tier 3 Paper-grade', vram_min=80, time_hr=22.0, model='Qwen3.6-27B',
             notebook='03_papergrade_qwen36_27b_cloud.ipynb', d_model=5120, layer=31,
             cost_usd=40.0, emoji='📜',
             url='https://github.com/OpenInterpretability/notebooks/blob/main/notebooks/03_papergrade_qwen36_27b_cloud.ipynb'),
    ]
    fit = [t for t in TIERS if t['vram_min'] <= vram_gb and t['time_hr'] <= time_budget]
    return fit[-1] if fit else TIERS[0]


def sae_memory(d_in, d_sae, fp32=True):
    """SAE param count + memory. Encoder + decoder + biases."""
    param_count    = 2 * d_in * d_sae + d_sae + d_in
    bytes_per_param = 4 if fp32 else 2
    params_mb      = param_count * bytes_per_param / 1e6
    activation_mb  = d_sae * 4 * 2 / 1e6  # bias + dec. negligible
    return dict(params_mb=params_mb, param_count=param_count, activation_mb=activation_mb)


pick = recommend(vram_gb, TRAINING_TIME_BUDGET_HOURS, MODEL_FAMILY_PREF, DIFFICULTY)
cost_str = 'free' if pick['cost_usd'] == 0 else f"~${pick['cost_usd']:.0f} cloud GPU"

print('=' * 58)
print(f"{pick['emoji']}  RECOMMENDATION: {pick['name']}")
print('=' * 58)
print(f"Model:          {pick['model']}")
print(f"Notebook:       {pick['notebook']}")
print(f"Open in Colab:  {pick['url']}")
print(f"Est. time:      ~{pick['time_hr']} h")
print(f"Est. cost:      {cost_str}")
print(f"Needs >=:       {pick['vram_min']} GB VRAM  (you have ~{vram_gb:.0f} GB)")
print()

# Mid-layer heuristic — applies beyond our 3 tiers
print('🎯 Mid-layer heuristic')
print(f"   Recommended layer for this tier: L{pick['layer']}")
print('   If your model differs from ours, pick roughly the middle residual layer —')
print('   it holds rich-but-not-too-specialized features. (recommended_layer = n_layers // 2)')
print()

# SAE VRAM calculator for the recommended tier
d_in  = pick['d_model']
d_sae = d_in * 16   # typical expansion factor
mem   = sae_memory(d_in, d_sae, fp32=True)
print('📐 SAE memory budget (expansion x16, fp32)')
print(f"   d_in = {d_in},  d_sae = {d_sae}")
print(f"   params:      {mem['param_count']/1e6:.1f} M  ({mem['params_mb']:.0f} MB fp32, {mem['params_mb']/2:.0f} MB bf16)")
print(f"   activations: ~{mem['activation_mb']:.2f} MB  (negligible)")
print()
print('Next step: open the Colab link above and run it.')